# Conversión del dataset ArchiveII
```
from pd.DataFrame   --> bpseqs/id_seqs.bpseq
                    --> fastas/id_seqs.fasta
```
-  Nota: convierte cada id del df temporalmente a ct y luego transforma.

In [65]:
import pandas as pd

In [66]:
REPO_PATH = "/home/gkulemeyer/Documents/Repos/RNA-analysis/DataAnalysis/"
# INPUT
DATA_PATH = REPO_PATH + "data/sources/ArchiveII.csv"
PARTITIONS_FOLDER = REPO_PATH + "data/filtered_dataset/data_partitions/valid_frac20/"

# OUTPUT
BPSEQ_PATH = REPO_PATH + "Formatting_csv_bpseq_fasta/usecase_ArchiveII/bpseqs"
FASTA_PATH = REPO_PATH + "Formatting_csv_bpseq_fasta/usecase_ArchiveII/fastas"

In [67]:
df = pd.read_csv(DATA_PATH)

df.set_index("id", inplace=True)
display(df.head())

,sequence,structure,base_pairs,len
id,,,,
5s_Acholeplasma-laidlawii-1,UCUGGUGACGAUAGGUAAGAUGGUUCACCUGUUCCCAUCCCGAACA...,((((((((......((((((((....((((((.............)...,"[[1, 111], [2, 110], [3, 109], [4, 108], [5, 1...",112
5s_Acidovorax-temperans-1,UGCCUGAUGACCAUAGCAAGUUGGUACCACUCCUUCCCAUCCCGAA...,.(((((((((.....((((((((.....((((((...............,"[[2, 115], [3, 114], [4, 113], [5, 112], [6, 1...",115
tmRNA_Stre.gord._TRW-29390_1-349,GGGGUCGUUACGGAUUCGACAGGCAUUAUGAGGCAUAUUUUGCGAC...,(((((((............((((((((....(((((((((..((((...,"[[1, 345], [2, 344], [3, 343], [4, 342], [5, 3...",349
tRNA_tdbR00000055-Schizosaccharomyces_pombe-4896-Glu-3UC,UCCGUUGUGGUCCAACGGCUAGGAUUCGUCGCUUUCACCGACGGGA...,(((((((..((((........))))((((((.......)))))).....,"[[1, 71], [2, 70], [3, 69], [4, 68], [5, 67], ...",75
srp_List.mono._U15684,UGGGUUGAUGAGCGUGAAGCCUUCGCUCGGUUGGAUUUUUCUUCAU...,.(.((((...(.(.((.(.((..(.....)..)).)...(...(.....,"[[2, 276], [4, 274], [5, 273], [6, 272], [7, 2...",279


In [68]:
def write_ct(fname, seqid, seq, base_pairs):
    """
    Write an RNA secondary structure in CT (Connectivity Table) format.

    Args:
        fname (str): Output CT file path.
        seqid (str): Sequence identifier.
        seq (str): RNA sequence.
        base_pairs (list[tuple[int, int]]): Base pairs (1-based), unique per nucleotide.

    Notes:
        The CT file is saved with the standard 6-column format:
        index, base, prev_index, next_index, pair_index, index_duplicate.
        Unpaired nucleotides have pair_index = 0.
    """
    base_pairs_dict = {}
    for bp in base_pairs:
        base_pairs_dict[bp[0]] = bp[1]
        base_pairs_dict[bp[1]] = bp[0]

    with open(fname, "w") as fout:
        fout.write(f"{len(seq)} {seqid}\n")
        for k, n in enumerate(seq):
            fout.write(f"{k+1} {n} {k} {k+2} {base_pairs_dict.get(k+1, 0)} {k+1}\n")


def remove_non_canonical(base_pairs, seq):
    """
    Remove non-canonical base pairs from a list of base pairs
    Args:
        base_pairs (list[tuple[int, int]]): Base pairs (1-based).
        seq (str): RNA sequence.

    Returns:
        list[tuple[int, int]]: Base pairs that are AU, UA, CG, GC, GU, or UG.
    """
    canonical = ["AU", "UA", "CG", "GC", "GU", "UG"]

    bp_canonical = []
    for bp in base_pairs:
        if seq[bp[0] - 1] + seq[bp[1] - 1] in canonical:
            bp_canonical.append(bp)

    return bp_canonical

In [69]:
# Convert a CT file with RNA sequence and structure to Bpseq format
def ct_to_bpseq(ct_file_path, bpseq_file_path):
    """
    Converts a CT file with RNA sequence and structure to Bpseq format.
    The Bpseq format is written to a file specified by `bpseq_file_path`.

    Args:
    - ct_file_path (str): Path to the input CT file.
    - bpseq_file_path (str): Path to the output Bpseq file.
    """
    with open(ct_file_path, "r") as ct_file, open(bpseq_file_path, "w") as bpseq_file:
        # Read the first line of the CT file to get the sequence length
        line = ct_file.readline()
        seq_length = int(line.strip().split()[0])

        # Initialize lists for the nucleotide names, positions, and base pairs
        nucleotide_names = [None] * (seq_length + 1)  # use None for the 0th position
        nucleotide_positions = [None] * (seq_length + 1)
        base_pairs = [None] * (seq_length + 1)

        # Read the nucleotide information from the CT file
        for i in range(seq_length):
            line = ct_file.readline()
            fields = line.strip().split()
            nucleotide_position = int(fields[0])
            nucleotide_name = fields[1]
            base_pair = int(fields[4])
            nucleotide_names[nucleotide_position] = nucleotide_name
            nucleotide_positions[nucleotide_position] = nucleotide_position
            base_pairs[nucleotide_position] = base_pair

        # Write the nucleotide information in Bpseq format to the output file
        for i in range(1, seq_length + 1):
            # ddd            bpseq_file.write(f"{i} {nucleotide_names[i]} {nucleotide_positions[i]} {base_pairs[i]}\n")
            bpseq_file.write(f"{i} {nucleotide_names[i]} {base_pairs[i]}\n")

In [70]:
df.index

Index(['5s_Acholeplasma-laidlawii-1', '5s_Acidovorax-temperans-1',
       'tmRNA_Stre.gord._TRW-29390_1-349',
       'tRNA_tdbR00000055-Schizosaccharomyces_pombe-4896-Glu-3UC',
       'srp_List.mono._U15684', '5s_Methanothermobacter-thermautotrophicus-6',
       'srp_Vibr.fisc._CP000020', 'srp_Baci.thur._D11412',
       'grp1_a.I1.m.M.grisea.B2.ND1', '5s_Saprospira-grandis-1',
       ...
       'RNaseP_M.avium', 'tRNA_tdbR00000521-Bos_taurus-9913-Ini-CAU',
       'grp1_a.I1.e.P.pachydermus.C1.SSU.943', '5s_Pseudomonas-stutzeri-2',
       '16s_T.maritima_domain3', '5s_Bacillus-cereus-6',
       'srp_Myco.aviu._AE016958', 'tmRNA_Heli.pylo._AE001503_1-383',
       '5s_Triticum-aestivum-1', '5s_Streptomyces-violaceus-1'],
      dtype='object', name='id', length=3864)

In [71]:
high = []
s = 0
for id in df.index:
    s += df.loc[id]["sequence"].count("N")
    if len(df.loc[id]["sequence"]) > 512:
        high.append(id)
print(s)
print(high)

0
[]


In [72]:
# SAVE FASTA
dirp = FASTA_PATH + "/"
if not os.path.exists(dirp):
    os.makedirs(dirp)

seq_ids = list(df.index)

for seq_id in seq_ids:
    if df.loc[seq_id].len <= 512:
        seqA = df.loc[seq_id]["sequence"].replace("N", "A")
        # igual da ERROR porque quedan pares incorrectos como AG
        seqA = seqA.replace("T", "U")
        # PDB-RNA has a T instead of U in sequence 3d2v-1-A! 😤
        if df.loc[seq_id]["sequence"].count("N") == 0:  # <<<<<<<<<<<<< ELIMINO
            with open(dirp + seq_id + ".fasta", "w") as tmpfile:
                tmpfile.write(">" + seq_id + "\n" + seqA + "\n")
    else:
        print("WARNING L>512:", seq_id)

In [73]:
!ls $FASTA_PATH 2>/dev/null | head -n 3
!ls $FASTA_PATH 2>/dev/null | wc -l
!cat "$FASTA_PATH"/$(ls "$FASTA_PATH" 2>/dev/null | head -n 1)


16s_A.fulgidus_domain2.fasta
16s_A.fulgidus_domain3.fasta
16s_A.fulgidus_domain4.fasta


3864
>16s_A.fulgidus_domain2
UUUAUUGGGCCUAAAGCGUCCGUAGCCGGGCUGGUAAGUCCUCCGGGAAAUCUGGCGGCUUAACCGUCAGACUGCCGGAGGAUACUGCCAGCCUAGGGACCGGGAGAGGCCGGGGGUAUUCCCGGAGUAGGGGUGAAAUCCUGUAAUCCCGGGAGGACCACCUGUGGCGAAGGCGCCCGGCUGGAACGGGUCCGACGGUGAGGGACGAAGGCCAGGGGAGCGAACCGGAUUAGAUACCCGGGUAGUCCUGGCUGUAAACGAUGCGGACUAGGUGUCACCGAAGCUACGAGCUUCGGUGGUGCCGGAGGGAAGCCGUUAAGUCCGCCGCCUGGGGAGUACGGCCGCAAGGCUGAAACUUA


In [74]:
# SAVE BSEQ

dirp = BPSEQ_PATH + "/"
if not os.path.exists(dirp):
    os.makedirs(dirp)

seq_ids = list(df.index)

for seq_id in seq_ids:
    if df.loc[seq_id].len <= 512:
        seqA = df.loc[seq_id]["sequence"].replace("N", "A")
        # igual da ERROR porque quedan pares incorrectos como AG
        seqA = seqA.replace("T", "U")
        # PDB-RNA has a T instead of U in sequence 3d2v-1-A! 😤
        if df.loc[seq_id]["sequence"].count("N") == 0:  # <<<<<<<<<<<<< ELIMINO
            bp_canonical = remove_non_canonical(
                ast.literal_eval(df.loc[seq_id]["base_pairs"]), seqA
            )
            write_ct("tmp.ct", seq_id, seqA, bp_canonical)
            # the use of intermediate CT file was inherited from the original code using RNAstructure conversor
            ct_to_bpseq("tmp.ct", dirp + seq_id + ".bpseq")
    else:
        print("WARNING L>512:", seq_id)

In [75]:
!ls $BPSEQ_PATH 2>/dev/null| head -n 3
!ls $BPSEQ_PATH 2>/dev/null| wc -l
!ls "$BPSEQ_PATH" 2>/dev/null| head -n 1
!cat "$BPSEQ_PATH"/$(ls "$BPSEQ_PATH" 2>/dev/null| head -n 1) | head -n 10

16s_A.fulgidus_domain2.bpseq
16s_A.fulgidus_domain3.bpseq
16s_A.fulgidus_domain4.bpseq
3864
16s_A.fulgidus_domain2.bpseq
1 U 0
2 U 0
3 U 0
4 A 0
5 U 0
6 U 0
7 G 0
8 G 329
9 G 328
10 C 327
